## Recommender Systems with Collaborative Filtering

Collaborative Filtering is a popular technique used in recommender systems to predict user preferences based on the preferences of similar users. It can be divided into two main types: user-based collaborative filtering and item-based collaborative filtering.

In [ ]:
import pandas as pd

anime_ratings = {'Oregairu': [5, None, 3, None, 5, 4, None, 
                              2, None, 5, 4, None, 5, 3, None, 4, None, 2, None, 5], 
                 'Gotoubun': [4, None, 4, None, 5, 3, 2, 2, None, 4, 5, None, 
                              4, 2, None, 5, 2, 3, None, 4], 
                 'RentGF': [5, 4, None, 4, 5, None, 1, 3, None, None, 4, None, 
                            5, 3, 4, None, 2, 3, None, 5],
                 'SoloLeveling': [None, 5, 2, 4, None, 3, 5, None, 5, None, 2, 5, None, 
                                  4, 4, None, 5, None, 4, None], 
                 'ChainsawMan': [None, 4, 3, 5, None, 4, 5, None, 4, None, 3, 4, None, 
                                 3, 5, None, 5, None, 4, None],
                 'SwordArtOnline': [3, None, None, 4, 2, None, 4, 3, 5, 3, 
                                    None, 4, 2, None, 5, 3, 5, 4, 3, 2],
                 'ReZero': [None, 3, None, None, 1, 3, None, 3, 5, None, 2, 4, 
                            None, 3, 4, None, 4, 4, 3, None],
                 'OnePiece': [2, 4, None, 5, None, None, 4, 4, None, 2, None, 
                              3, None, 2, None, 3, 4, 4, None, None],
                 'Naruto': [3, 5, 2, 4, 1, 3, 5, 4, 5, 3, 2, 5, 1, None, 4, 3, 4, 4, 3, 2],
                 'DragonBall': [2, None, 1, None, 2, 4, 3, 3, 4, None, 2, 4, 2, 2, 3, 2, 3, 3, 4, 3]}
# Mean Normalization
df = pd.DataFrame(anime_ratings)
column_mean = df.mean(axis=0)
new_df = df.subtract(column_mean)
new_df = new_df.fillna(0)

In [44]:
from sklearn.metrics.pairwise import cosine_similarity
# Compute cosine similarity between all anime vectors (rows)
# Note: Transpose the matrix first
new_df = new_df.T
similarity_matrix = cosine_similarity(new_df)
anime_names = df.columns.tolist()
# Convert to DataFrame for readable table
similarity_df = pd.DataFrame(similarity_matrix, index=anime_names, 
                             columns=anime_names)
print(similarity_df.shape)

(10, 10)


In [45]:
# Find those who have rated SoloLeveling
user_df = new_df.T
target_anime = 'SoloLeveling'
watched_users = user_df[user_df[target_anime] != 0].index.tolist()
# Target Vectors
target_user_vector = user_df.iloc[0].values.reshape(1, -1) # U1
similarities = []
for i in watched_users:
  if i == 0: # skip comparing U1 with itself
    continue
  sim = cosine_similarity(target_user_vector,
  user_df.iloc[i].values.reshape(1, -1))[0][0]
  similarities.append((i, sim))
# Sort by descending similarity
similarities.sort(key=lambda x: x[1], reverse=True)
top_k = similarities[:3]
print(f"Top 3 similar users to U1 who rated {target_anime}:")
for i, sim in top_k:
  print(f"User {i + 1} → Similarity: {sim}")

Top 3 similar users to U1 who rated SoloLeveling:
User 11 → Similarity: 0.2735474826196078
User 3 → Similarity: 0.12627609433321602
User 2 → Similarity: -0.1956790189148629


In [46]:
# Final Rating Prediction
numerator = 0
denominator = 0
for i, sim in top_k:
  rating = user_df.iloc[i][target_anime] # mean-centered rating
  numerator += sim * rating
  denominator += abs(sim)
if denominator != 0:
  predicted_normalized_rating = numerator / denominator
else:
  predicted_normalized_rating = 0 # fallback if no similar users

print(f"Predicted normalized rating of U1 for {target_anime}: {predicted_normalized_rating}")
# Add back the column mean
real_predicted_rating = predicted_normalized_rating + column_mean[target_anime]
print(f"Predicted real rating of U1 for {target_anime}: {real_predicted_rating}")

Predicted normalized rating of U1 for SoloLeveling: -1.6714052629279548
Predicted real rating of U1 for SoloLeveling: 2.328594737072045


In [47]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def user_based_recommend_systems(new_df, user, similar_users=2):
    target_anime = df.columns.tolist()
    # Find those who have rated the anime
    user_df = new_df.T
    for j in target_anime:
        watched_users = user_df[user_df[j] != 0].index.tolist()
        # Target Vectors
        target_user_vector = user_df.iloc[user].values.reshape(1, -1)
        similarities = []
        for i in watched_users:
            if i == user:  # skip comparing the user with itself
                continue
            sim = cosine_similarity(target_user_vector, user_df.iloc[i].values.reshape(1, -1))[0][0]
            similarities.append((i, sim))
        # Sort by descending similarity
        similarities.sort(key=lambda x: x[1], reverse=True)
        top_k = similarities[: similar_users]

        # Final Rating Prediction
        numerator = 0
        denominator = 0
        for i, sim in top_k:
            rating = user_df.iloc[i][j]  # mean-centered rating
            numerator += sim * rating
            denominator += abs(sim)
        if denominator != 0:
            predicted_normalized_rating = numerator / denominator
        else:
            predicted_normalized_rating = 0  # fallback if no similar users
        # Add back the column mean
        real_predicted_rating = predicted_normalized_rating + column_mean[j]
        print(f"Predicted real rating of user with index {user} for {j}: {real_predicted_rating}")

In [48]:
def item_based_recommend_systems(new_df, user_index, similar_items=3):
    # new_df: rows = anime, columns = users (mean-centered)
    target_user_ratings = new_df.iloc[:, user_index]
    for anime in new_df.index:
        # Skip anime already rated by the user
        if target_user_ratings[anime] != 0:
            continue
        similarities = []
        # Compare target anime with other anime
        target_anime_vector = new_df.loc[anime].values.reshape(1, -1)
        for other_anime in new_df.index:
            if other_anime == anime:
                continue
            # Only consider anime the user has rated
            if target_user_ratings[other_anime] == 0:
                continue
            sim = cosine_similarity(target_anime_vector,
                                    new_df.loc[other_anime].values.reshape(1, -1))[0][0]
            similarities.append((other_anime, sim))
        # Select top-K similar anime
        similarities.sort(key=lambda x: x[1], reverse=True)
        top_k = similarities[:similar_items]
        # Predict normalized rating
        numerator = 0
        denominator = 0
        for other_anime, sim in top_k:
            numerator += sim * target_user_ratings[other_anime]
            denominator += abs(sim)
        if denominator != 0:
            predicted_normalized_rating = numerator / denominator
        else:
            predicted_normalized_rating = 0

        # Add back anime mean
        real_rating = predicted_normalized_rating + column_mean[anime]
        print(f"Predicted real rating for User {user_index} on {anime}: {real_rating}")

In [49]:
user = 2
user_based_recommend_systems(new_df, user, similar_users=2)

Predicted real rating of user with index 2 for Oregairu: 4.0
Predicted real rating of user with index 2 for Gotoubun: 5.0
Predicted real rating of user with index 2 for RentGF: 4.2919500272825655
Predicted real rating of user with index 2 for SoloLeveling: 2.0003496200630533
Predicted real rating of user with index 2 for ChainsawMan: 3.0
Predicted real rating of user with index 2 for SwordArtOnline: 2.552177915831489
Predicted real rating of user with index 2 for ReZero: 1.7313683526729176
Predicted real rating of user with index 2 for OnePiece: 2.5806930147289924
Predicted real rating of user with index 2 for Naruto: 2.3370523562455072
Predicted real rating of user with index 2 for DragonBall: 2.0


In [50]:
user_index = 4
item_based_recommend_systems(new_df, user_index, similar_items=3)

Predicted real rating for User 4 on SoloLeveling: 2.196036637741274
Predicted real rating for User 4 on ChainsawMan: 2.5591853317993882
Predicted real rating for User 4 on OnePiece: 1.7856274075682537
